# Transcribe the model rating into the scoring form

Reads the raw session transcripts in `Data/Reliability/raw/`, maps the verdicts through the
fixed lookup table, applies the quotation gate as amended by deviation D13, and writes the
result into `Handover/scoring_form.xlsx` for `rater_agreement.ipynb` to read.

**This notebook never loads rater 1's verdicts.** It does not read `scoring_long.csv` and it
does not touch the tracker. That is deliberate. Every judgment made here has to be made without
knowing whether it moves agreement up or down, and the cleanest way to guarantee that is for the
information not to be present at all.

Run it in two stages. Stage one parses, gates and exports a blind adjudication list. You fill
that list in. Stage two reads it back and writes the form. Stage two refuses to run until every
flagged cell has a decision.

---

## The rules, fixed before the first run

**Verdict mapping.** Appendix B of `Drafts/fallback_rater_protocol.md`. Anything the table does
not cover is escalated and is never guessed.

**Quotation gate, per D13.** A verdict asserting something is *present* in the paper needs a
verbatim quotation and becomes `Unclear` without one. A verdict asserting something is *absent*
is exempt, because absence cannot be quoted.

**What counts as a quotation.** The model wraps section names in quotation marks as freely as it
wraps quoted text, so the presence of quote characters proves nothing. A quoted span counts as a
quotation only if it runs to eight or more words, or if it ends in sentence punctuation inside
the quotation marks. Section labels such as `"Randomization"` satisfy neither. This rule is
fixed here and is not tuned after seeing which cells it catches. A parsing bug may be fixed; the
rule may not.

**Blind adjudication.** Cells with no qualifying quotation are exported with the item, the note
and the model's verdict, and without rater 1's verdict. The only question asked is whether the
verdict asserts presence or absence.


In [ ]:
import re
from pathlib import Path

import pandas as pd

ROOT  = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
RUN   = 1        # only the primary run is transcribed, fixed in advance by D12
RAW   = ROOT / 'Data' / 'Reliability' / 'Raw'
ADJ   = ROOT / 'Data' / 'Reliability' / 'adjudication_blind.csv'
TPL   = ROOT / 'Handover' / 'scoring_form.xlsx'      # blank, ships to raters, never written
FORM  = ROOT / 'Data' / 'Reliability' / f'rating_model_run{RUN}.xlsx'


# --- Appendix B lookup, applied mechanically -----------------------------
VERDICT = {'yes': 'Yes', 'no': 'No',
           'unclear': 'Unclear', 'uncertain': 'Unclear', 'cannot determine': 'Unclear',
           'not reported': 'Unclear', 'insufficient information': 'Unclear',
           'n/a': 'N/A', 'na': 'N/A', 'n.a.': 'N/A', 'not applicable': 'N/A'}
CONFIDENCE = {'high': 'High', 'medium': 'Medium', 'low': 'Low'}

def norm(cell):
    """Strip html tags, then markdown emphasis, collapse whitespace, lowercase.

    Tag removal has to come first. Stripping angle brackets as bare characters would leave the
    body of a <br> behind as the word 'br', which is how an entire paper's verdicts once
    escalated instead of mapping.
    """
    s = re.sub(r'<[^>]*>', ' ', str(cell))
    s = re.sub(r'[*`_]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip().lower()

def transcripts(run):
    """Find a run's transcripts under either the flat or the per run folder layout."""
    found = sorted(RAW.rglob(f'run{run}_paper*.txt'))
    empty = [p.name for p in found if p.stat().st_size == 0]
    assert not empty, f'empty transcript file(s), nothing to parse: {empty}'
    return found

print('raw transcripts:', [p.name for p in transcripts(RUN)])


## Parse

The table shape is not stable between sessions. Some transcripts carry a separate item number
column, some merge the number into the item name, some omit the name entirely. Columns are
therefore located by header text rather than by position.

A logical row may span several physical lines, because the model emits `<br>` and real newlines
inside note cells. A new row is recognised by a line beginning with a pipe whose first cell
starts with a digit, which is true of every table shape seen and false of every continuation
line, header and separator.


In [ ]:
ROW_START = re.compile(r'^\s*\|\s*\**\s*\d')

def parse_transcript(path):
    lines = path.read_text(encoding='utf-8').split('\n')
    header, rows, cur, started = None, [], None, False
    for ln in lines:
        s = ln.strip()
        if not started:
            if s.startswith('|') and re.search(r'verdict', s, re.I):
                header, started = s, True
            continue
        if re.match(r'^(---|##)', s) and cur is not None and not s.startswith('|'):
            break          # end of table
        if ROW_START.match(ln):
            if cur is not None:
                rows.append(cur)
            cur = s
        elif cur is not None:
            cur += ' ' + s
    if cur is not None:
        rows.append(cur)

    hdr = [norm(c) for c in header.strip().strip('|').split('|')]
    def col(*keys):
        for i, h in enumerate(hdr):
            if any(k in h for k in keys):
                return i
        raise KeyError(f'no column matching {keys} in {hdr}')
    iv, ic, inote = col('verdict'), col('confid'), col('note')

    out = []
    for r in rows:
        cells = [c.strip() for c in r.strip().strip('|').split('|')]
        if len(cells) != len(hdr):
            out.append({'parse_error': f'{len(cells)} cells, header has {len(hdr)}', 'raw': r})
            continue
        m = re.search(r'\d+', cells[0])
        out.append({'item_no': int(m.group()) if m else None,
                    'verdict_raw': cells[iv], 'confidence_raw': cells[ic],
                    'note': cells[inote], 'parse_error': None})
    return out

records = []
for p in transcripts(RUN):
    pid = int(re.search(r'paper(\d+)', p.name).group(1))
    for rec in parse_transcript(p):
        records.append({'paper_id': pid, 'source': p.name, **rec})

df = pd.DataFrame(records)
errs = df[df.parse_error.notna()]
assert errs.empty, f'parse errors, resolve before continuing:\n{errs}'
assert len(df) == 65, f'expected 65 cells, parsed {len(df)}'
assert df.groupby('paper_id').item_no.count().eq(13).all(), 'not 13 items per paper'
assert df.groupby('paper_id').item_no.apply(lambda s: sorted(s) == list(range(1, 14))).all(), \
    'item numbers are not 1..13 in every paper'
print(f'parsed {len(df)} cells across {df.paper_id.nunique()} papers, no parse errors')


## Map the verdicts

Mechanical. Anything the lookup does not cover is escalated rather than interpreted, because
this is the step at which a disagreement could quietly be typed into an agreement.


In [ ]:
df['verdict'] = df.verdict_raw.map(lambda c: VERDICT.get(norm(c)))
df['confidence'] = df.confidence_raw.map(lambda c: CONFIDENCE.get(norm(c)))

esc = df[df.verdict.isna() | df.confidence.isna()]
if len(esc):
    print(f'ESCALATION, {len(esc)} cell(s) the lookup does not cover. Do not guess these.')
    for _, r in esc.iterrows():
        print(f"  paper {r.paper_id} item {r.item_no}: verdict={r.verdict_raw!r} "
              f"confidence={r.confidence_raw!r}")
else:
    print('all 65 cells mapped cleanly, no escalations')

print()
print(df.verdict.value_counts().to_string())
print()
print(df.confidence.value_counts().to_string())


## Apply the quotation gate

`N/A` and `Unclear` were always exempt. Under D13 the exemption extends to any verdict resting
on absence, so what is flagged here is every `Yes` and `No` carrying no qualifying quotation.
Those are the cells that need a presence or absence call, and that call is made blind.


In [ ]:
QUOTED = re.compile(r'["\u201c]([^"\u201d]{2,}?)["\u201d]')

def has_quotation(note):
    """A quoted span counts only if it is a sentence, not a section label."""
    for m in QUOTED.finditer(str(note)):
        span = m.group(1).strip()
        if len(span.split()) >= 8:
            return True
        if re.search(r'[.!?]$', span):
            return True
    return False

df['quoted'] = df.note.map(has_quotation)
df['gate_applies'] = df.verdict.isin(['Yes', 'No'])
df['flagged'] = df.gate_applies & ~df.quoted

print(f"cells subject to the gate : {int(df.gate_applies.sum())}")
print(f"  carrying a quotation    : {int((df.gate_applies & df.quoted).sum())}")
print(f"  flagged for adjudication: {int(df.flagged.sum())}")
print(f"exempt as N/A or Unclear  : {int((~df.gate_applies).sum())}")
print()
print(df[df.flagged].groupby('paper_id').size().rename('flagged').to_string())


## Export the blind adjudication list

One row per flagged cell. Fill the `CALL` column with `PRESENCE` or `ABSENCE` and give a short
reason. `PRESENCE` means the verdict asserts that something is in the paper, so the missing
quotation converts it to `Unclear`. `ABSENCE` means the verdict rests on something not being
there, which cannot be quoted, so the verdict stands.

Rater 1's verdicts are not in this file and must not be consulted while filling it in.


In [ ]:
cols = ['paper_id', 'item_no', 'model_verdict', 'model_confidence', 'note', 'CALL', 'reason']
if ADJ.exists():
    print(f'{ADJ.name} already exists, not overwriting. Delete it if you mean to start over.')
else:
    out = (df[df.flagged]
           .assign(model_verdict=lambda d: d.verdict, model_confidence=lambda d: d.confidence,
                   CALL='', reason='')
           .sort_values(['paper_id', 'item_no'])[cols])
    out.to_csv(ADJ, index=False, encoding='utf-8')
    print(f'wrote {ADJ.relative_to(ROOT)} with {len(out)} cell(s) to adjudicate')


---

## Stage two, write the form

Runs only once every flagged cell has a decision. `PRESENCE` cells become `Unclear` and keep
their original verdict in the note, so the conversion is visible rather than silent. Everything
else is written as rated.


In [ ]:
if not ADJ.exists():
    print('no adjudication file yet, stage one first')
else:
    adj = pd.read_csv(ADJ, keep_default_na=False)
    blank = adj[~adj.CALL.str.upper().isin(['PRESENCE', 'ABSENCE'])]
    if len(blank):
        print(f'{len(blank)} cell(s) still undecided, form not written:')
        print(blank[['paper_id', 'item_no', 'CALL']].to_string(index=False))
    else:
        calls = {(int(r.paper_id), int(r.item_no)): r.CALL.upper() for _, r in adj.iterrows()}
        final = df.copy()
        final['final_verdict'] = final.verdict
        final['gate_note'] = ''
        for i, r in final.iterrows():
            key = (int(r.paper_id), int(r.item_no))
            if key in calls and calls[key] == 'PRESENCE':
                final.at[i, 'final_verdict'] = 'Unclear'
                final.at[i, 'gate_note'] = (
                    f' [QUOTATION GATE, D13: rated {r.verdict} with no verbatim quotation for a '
                    f'presence based claim, recorded Unclear by rule.]')
        n_flip = int((final.final_verdict != final.verdict).sum())
        print(f'{n_flip} verdict(s) converted to Unclear by the gate')

        import openpyxl, shutil
        # Always start from the blank template. Writing into the handover copy would
        # hand a future rater the answers, which happened once and was caught late.
        shutil.copy(TPL, FORM)
        wb = openpyxl.load_workbook(FORM)
        ws = wb['Scoring']
        idx = {(int(r[0].value), int(r[2].value)): r for r in ws.iter_rows(min_row=2)
               if r[0].value is not None}
        written = 0
        for _, r in final.iterrows():
            row = idx[(int(r.paper_id), int(r.item_no))]
            row[5].value = r.final_verdict                      # VERDICT
            row[6].value = r.confidence                         # CONFIDENCE
            row[7].value = str(r.note) + r.gate_note            # NOTE
            written += 1
        wb.save(FORM)
        print(f'wrote {written} cells to {FORM.relative_to(ROOT)}')
        print('next: Run All on rater_agreement.ipynb, once, before any discussion')
